In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
# Neural Networks comprise of layers/modules that perform operations on data. 

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10),
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    

In [6]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [7]:
X = torch.rand(1, 28, 28, device = device)
logits = model(X)
pred_probab = nn.Softmax(dim = 1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([0], device='mps:0')


In [8]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())

torch.Size([3, 28, 28])


In [10]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [12]:
layer1 = nn.Linear(in_features = 28*28, out_features = 20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [13]:
print(f"Before ReLU: {hidden1} \n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.2317,  0.2909,  0.1491, -0.3233, -0.0537,  0.0087,  0.1026,  0.1177,
          0.2195,  0.0414,  0.0753, -0.3672,  0.1018, -0.1426, -0.4827, -0.3334,
         -0.0041,  0.2658,  0.3986, -0.4968],
        [ 0.0566, -0.3033,  0.0539,  0.0024,  0.0879,  0.3636, -0.1722,  0.5241,
          0.0656,  0.1388,  0.2867, -0.2893,  0.1408,  0.1811, -0.4200, -0.0502,
          0.0147,  0.2039,  0.1994, -0.0277],
        [ 0.3551,  0.2466,  0.0716, -0.3266,  0.1794,  0.2208,  0.3211,  0.5134,
          0.0349,  0.0808,  0.4309, -0.4619,  0.4223, -0.2231, -0.1954, -0.1037,
          0.0248,  0.3034,  0.1058, -0.2974]], grad_fn=<AddmmBackward0>) 


After ReLU: tensor([[0.2317, 0.2909, 0.1491, 0.0000, 0.0000, 0.0087, 0.1026, 0.1177, 0.2195,
         0.0414, 0.0753, 0.0000, 0.1018, 0.0000, 0.0000, 0.0000, 0.0000, 0.2658,
         0.3986, 0.0000],
        [0.0566, 0.0000, 0.0539, 0.0024, 0.0879, 0.3636, 0.0000, 0.5241, 0.0656,
         0.1388, 0.2867, 0.0000, 0.1408, 0.1811, 0.0

In [14]:
seq_models = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20,10)
)
input_image = torch.rand(3, 28, 28)
logits = seq_models(input_image)

In [15]:
softmax = nn.Softmax(dim = 1)
pred_probab = softmax(logits)

In [16]:
print(f"Model structure: {model}\n\n")
for name, param in model.named_parameters():
    print(f"Layer: {name}, Size: {param.size()}, Values: {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight, Size: torch.Size([512, 784]), Values: tensor([[-0.0321,  0.0035,  0.0091,  ..., -0.0099,  0.0003,  0.0288],
        [ 0.0100, -0.0026, -0.0071,  ...,  0.0240,  0.0088, -0.0330]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias, Size: torch.Size([512]), Values: tensor([0.0147, 0.0274], device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight, Size: torch.Size([512, 512]), Values: tensor([[-0.0322, -0.0251,  0.0239,  ...,  0.0233, -0.0218,  0.0109],
        [-0.0103, -0.0402,  0.0076,  ...,  0.0427, -0.0369, -0.0040]],
       device='mps:0', grad_fn=<SliceBackward0>)